# Exploratory Data Analysis — IBM AML HI-Small

**Purpose:** Characterize the dataset before modeling. Understand class imbalance, temporal patterns, account connectivity, transaction amounts, and categorical features. Every finding here directly informs a modeling decision.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import defaultdict

sys.path.insert(0, ".")
from src.data.loader import load_raw_data
from src.utils.config import DataConfig

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1, palette="Set2")
COLOR_LEGIT = "#4C72B0"
COLOR_LAUNDER = "#C44E52"
FIGSIZE_WIDE = (7, 3.5)
DPI = 120
OUT = Path("temp/eda")
OUT.mkdir(parents=True, exist_ok=True)

%matplotlib inline

In [ ]:
# Load the dataset
cfg = DataConfig(dataset_variant="HI-Small")
accounts, txn = load_raw_data(cfg)

n_txn = len(txn)
n_laundering = txn["is_laundering"].sum()
n_accounts = len(accounts)
prevalence = n_laundering / n_txn

ts_dt = pd.to_datetime(txn["timestamp"], errors="coerce")
ts_days = (ts_dt.max() - ts_dt.min()).total_seconds() / 86400

print(f"Accounts: {n_accounts:,}")
print(f"Transactions: {n_txn:,}")
print(f"Laundering transactions: {n_laundering:,} ({prevalence:.4%})")
print(f"Time span: {ts_days:.1f} days")
print(f"Graph density: {n_txn / (n_accounts * (n_accounts - 1)):.8f}")

**Key numbers at a glance:** 518K accounts, 5.08M transactions across 17.7 days, 0.102% laundering prevalence. The graph is extremely sparse (density ~1.9e-8). A random classifier would achieve AUC-ROC ~0.5 and AUC-PR ~0.001 (the prevalence rate).

## 1. Class Imbalance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGSIZE_WIDE, dpi=DPI)

counts = [n_txn - n_laundering, n_laundering]
labels = ["Legitimate", "Laundering"]
colors = [COLOR_LEGIT, COLOR_LAUNDER]

bars = ax1.bar(labels, counts, color=colors, edgecolor="white", linewidth=0.5)
ax1.set_ylabel("Transactions")
ax1.set_title("Class Distribution")
for bar, val in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + n_txn * 0.01,
             f"{val:,}\n({100*val/n_txn:.3f}%)", ha="center", fontsize=8)

ax2.pie(counts, labels=labels, colors=colors, autopct="%1.3f%%",
        startangle=90, explode=(0, 0.1), textprops={"fontsize": 9})
ax2.set_title("Class Proportion")

fig.suptitle("Figure 1: Class imbalance", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "01_class_imbalance.png", bbox_inches="tight")
plt.show()

Only 5,177 out of 5.08M transactions are laundering (0.102%). This is extreme class imbalance: the minority class is roughly 1 in 1,000. Accuracy is meaningless here. We report AUC-ROC, AUC-PR, Precision, Recall, and F1 instead. In training, we use `pos_weight` (ratio of negative to positive samples in the training set) in the binary cross-entropy loss to prevent the model from ignoring the minority class.

## 2. Temporal Distribution

In [ ]:
txn["timestamp_dt"] = pd.to_datetime(txn["timestamp"], unit="s", errors="coerce")
txn["hour"] = txn["timestamp_dt"].dt.hour
txn["day"] = txn["timestamp_dt"].dt.day
txn["dayofweek"] = txn["timestamp_dt"].dt.dayofweek

fig, axes = plt.subplots(2, 2, figsize=(8, 6), dpi=DPI)

# Hourly volume by class
for label_val, col, name in [(0, COLOR_LEGIT, "Legitimate"), (1, COLOR_LAUNDER, "Laundering")]:
    subset = txn[txn["is_laundering"] == label_val]
    hourly = subset.groupby("hour").size().reindex(range(24), fill_value=0)
    axes[0, 0].plot(hourly.index, hourly.values, color=col, label=name, linewidth=1.2)
axes[0, 0].set_xlabel("Hour of Day")
axes[0, 0].set_ylabel("Transactions")
axes[0, 0].set_title("Hourly Transaction Volume")
axes[0, 0].legend(fontsize=7)

# Daily volume stacked
daily = txn.groupby(["day", "is_laundering"]).size().unstack(fill_value=0)
axes[0, 1].bar(daily.index, daily.get(0, 0), color=COLOR_LEGIT, label="Legitimate", width=0.8)
axes[0, 1].bar(daily.index, daily.get(1, 0), color=COLOR_LAUNDER, label="Laundering",
              width=0.8, bottom=daily.get(0, 0))
axes[0, 1].set_xlabel("Day (within dataset)")
axes[0, 1].set_ylabel("Transactions")
axes[0, 1].set_title("Daily Volume (stacked)")
axes[0, 1].legend(fontsize=7)

# Laundering rate by day
daily_rate = (daily.get(1, 0) / daily.sum(axis=1) * 100)
axes[1, 0].bar(daily_rate.index, daily_rate.values, color=COLOR_LAUNDER, width=0.8)
axes[1, 0].axhline(y=prevalence * 100, color="black", linestyle="--", linewidth=0.8,
                   label=f"Overall ({prevalence*100:.3f}%)")
axes[1, 0].set_xlabel("Day")
axes[1, 0].set_ylabel("Laundering Rate (%)")
axes[1, 0].set_title("Laundering Rate by Day")
axes[1, 0].legend(fontsize=7)

# Day of week
dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
for label_val, col, name in [(0, COLOR_LEGIT, "Legitimate"), (1, COLOR_LAUNDER, "Laundering")]:
    subset = txn[txn["is_laundering"] == label_val]
    dow_counts = subset.groupby("dayofweek").size().reindex(range(7), fill_value=0)
    offset = -0.15 if label_val == 0 else 0.15
    axes[1, 1].bar(np.arange(7) + offset, dow_counts.values, width=0.3, color=col, label=name)
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(dow_names, fontsize=8)
axes[1, 1].set_ylabel("Transactions")
axes[1, 1].set_title("Volume by Day of Week")
axes[1, 1].legend(fontsize=7)

fig.suptitle("Figure 2: Temporal distribution of transactions", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "02_temporal_distribution.png", bbox_inches="tight")
plt.show()

Transactions follow a daily cycle with a dip around 9-10 AM and peak activity during business hours. The laundering rate varies across days (0.06% to 0.16%), showing that laundering activity is not uniformly distributed across time. This temporal non-uniformity supports the use of chronological train/test splits: if we randomly shuffle, the model sees laundering patterns from all time periods during training, which would not be available in a real deployment where only past data is known. Day-of-week patterns are similar across both classes, with slightly lower volume on weekends.

## 3. Transaction Amounts

In [ ]:
amt_legit = txn.loc[txn["is_laundering"] == 0, "amount"].clip(upper=1e6)
amt_launder = txn.loc[txn["is_laundering"] == 1, "amount"].clip(upper=1e6)

print(f"Median amount: legitimate = ${amt_legit.median():,.0f}, laundering = ${amt_launder.median():,.0f}")
print(f"Mean amount:   legitimate = ${amt_legit.mean():,.0f}, laundering = ${amt_launder.mean():,.0f}")
print(f"Std amount:    legitimate = ${amt_legit.std():,.0f}, laundering = ${amt_launder.std():,.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGSIZE_WIDE, dpi=DPI)

bins = np.logspace(0, 6, 80)
ax1.hist(amt_legit, bins=bins, color=COLOR_LEGIT, alpha=0.6, label="Legitimate", density=True)
ax1.hist(amt_launder, bins=bins, color=COLOR_LAUNDER, alpha=0.7, label="Laundering", density=True)
ax1.set_xscale("log")
ax1.set_xlabel("Amount (log scale)")
ax1.set_ylabel("Density")
ax1.set_title("Amount Distribution (log-log)")
ax1.legend(fontsize=8)

ax2.boxplot([amt_legit, amt_launder], tick_labels=["Legitimate", "Laundering"],
            patch_artist=True, boxprops={"facecolor": COLOR_LEGIT, "alpha": 0.5},
            medianprops={"color": "black"})
ax2.set_ylabel("Amount")
ax2.set_title("Amount by Class")

fig.suptitle("Figure 3: Transaction amount distributions", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "03_amount_distribution.png", bbox_inches="tight")
plt.show()

Laundering transactions have a substantially higher median amount ($8,667 vs $1,408). Both distributions are heavily right-skewed. However, there is significant overlap: many legitimate transactions are also large, and some laundering transactions are small. Amount alone is not sufficient for detection. This is consistent with real-world laundering typologies where structuring (smurfing) involves many small transactions below reporting thresholds, while integration often involves large single transfers. The feature engineering step applies log1p transformation to amount to handle the long-tailed distribution.

## 4. Account Degree Distributions

In [ ]:
out_deg = txn.groupby("from_account").size()
in_deg = txn.groupby("to_account").size()
total_deg = out_deg.add(in_deg, fill_value=0)

laundering_out = txn[txn["is_laundering"] == 1].groupby("from_account").size()
laundering_in = txn[txn["is_laundering"] == 1].groupby("to_account").size()
laundering_accts = set(laundering_out.index) | set(laundering_in.index)

all_accts = set(out_deg.index) | set(in_deg.index)
legit_deg = [total_deg.get(a, 0) for a in all_accts if a not in laundering_accts]
laund_deg = [total_deg.get(a, 0) for a in all_accts if a in laundering_accts]

print(f"Accounts with >=1 transaction: {len(all_accts):,}")
print(f"Accounts involved in laundering: {len(laundering_accts):,}")
print(f"Median total degree: legitimate = {np.median(legit_deg):.1f}, laundering = {np.median(laund_deg):.1f}")
print(f"Mean total degree:   legitimate = {np.mean(legit_deg):.1f}, laundering = {np.mean(laund_deg):.1f}")

fig, axes = plt.subplots(2, 2, figsize=(8, 6), dpi=DPI)

for ax, deg_series, title in [
    (axes[0, 0], out_deg, "Out-degree Distribution"),
    (axes[0, 1], in_deg, "In-degree Distribution"),
]:
    values = deg_series.values
    ax.hist(np.log10(values + 1), bins=60, color=COLOR_LEGIT, alpha=0.7, density=True)
    ax.set_xlabel("log10(degree + 1)")
    ax.set_ylabel("Density")
    ax.set_title(title)

axes[1, 0].hist([np.log10(np.array(legit_deg) + 1), np.log10(np.array(laund_deg) + 1)],
                bins=50, color=[COLOR_LEGIT, COLOR_LAUNDER], alpha=0.6,
                label=["Legitimate", "Laundering"], density=True)
axes[1, 0].set_xlabel("log10(total degree + 1)")
axes[1, 0].set_ylabel("Density")
axes[1, 0].set_title("Total Degree by Class")
axes[1, 0].legend(fontsize=7)

axes[1, 1].boxplot([legit_deg, laund_deg], tick_labels=["Legitimate", "Laundering"],
                   patch_artist=True,
                   boxprops={"facecolor": COLOR_LEGIT, "alpha": 0.5},
                   medianprops={"color": "black"})
axes[1, 1].set_ylabel("Total Degree")
axes[1, 1].set_title("Degree by Class (boxplot)")

fig.suptitle("Figure 4: Account degree distributions", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "04_degree_distributions.png", bbox_inches="tight")
plt.show()

Accounts involved in laundering have substantially higher connectivity (median degree 22.0 vs 6.0 for legitimate accounts). This is a structural signature of laundering: accounts participating in layering or fan-out schemes transact with many more counterparties than normal accounts. This elevated connectivity is exactly the kind of pattern that GNNs can exploit through neighborhood aggregation, but flat ML models cannot see. The degree distributions are heavy-tailed in both classes, following a power-law-like pattern typical of financial transaction networks.

## 5. Payment Format Analysis

In [ ]:
pmt_legit = txn[txn["is_laundering"] == 0]["payment_format"].value_counts()
pmt_launder = txn[txn["is_laundering"] == 1]["payment_format"].value_counts()

pmt_all = pd.DataFrame({"Legitimate": pmt_legit, "Laundering": pmt_launder}).fillna(0)
pmt_pct = pmt_all.div(pmt_all.sum(axis=0), axis=1) * 100
pmt_pct = pmt_pct.sort_values("Laundering", ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGSIZE_WIDE, dpi=DPI)

pmt_pct.plot(kind="barh", ax=ax1, color=[COLOR_LEGIT, COLOR_LAUNDER], width=0.8)
ax1.set_xlabel("% of Transactions")
ax1.set_title("Payment Format Usage by Class")
ax1.legend(fontsize=7)

pmt_launder_rate = (pmt_all["Laundering"] / pmt_all.sum(axis=1) * 100).sort_values(ascending=False)
ax2.barh(pmt_launder_rate.index, pmt_launder_rate.values, color=COLOR_LAUNDER)
ax2.axvline(x=prevalence * 100, color="black", linestyle="--", linewidth=0.8,
            label=f"Overall ({prevalence*100:.3f}%)")
ax2.set_xlabel("Laundering Rate (%)")
ax2.set_title("Laundering Rate by Payment Format")
ax2.legend(fontsize=7)

print("Laundering rate by payment format:")
for fmt, rate in pmt_launder_rate.items():
    print(f"  {fmt}: {rate:.3f}%")

fig.suptitle("Figure 5: Payment format analysis", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "05_payment_format.png", bbox_inches="tight")
plt.show()

ACH (Automated Clearing House) transactions have a laundering rate of 0.746%, roughly 7.3 times the overall prevalence of 0.102%. This is consistent with real-world laundering patterns: ACH enables batch processing and is commonly used for layering. All other payment formats (Bitcoin, Cash, Cheque, Credit Card) have laundering rates well below the overall average. Payment format is therefore a strong univariate signal and is included as a one-hot encoded feature in the edge feature matrix.

## 6. Graph Connectivity

In [ ]:
# Build undirected adjacency and find connected components
edges = list(zip(txn["from_account"], txn["to_account"]))
node_to_idx = {}
idx_to_node = []
for u, v in edges:
    for node in (u, v):
        if node not in node_to_idx:
            node_to_idx[node] = len(idx_to_node)
            idx_to_node.append(node)
n_nodes = len(idx_to_node)

adj = defaultdict(set)
for u, v in edges:
    ui, vi = node_to_idx[u], node_to_idx[v]
    adj[ui].add(vi)
    adj[vi].add(ui)

visited = set()
comp_sizes = []
for node in range(n_nodes):
    if node in visited:
        continue
    stack = [node]
    visited.add(node)
    size = 0
    while stack:
        cur = stack.pop()
        size += 1
        for nb in adj[cur]:
            if nb not in visited:
                visited.add(nb)
                stack.append(nb)
    comp_sizes.append(size)

comp_sizes.sort(reverse=True)
print(f"Graph nodes: {n_nodes:,}")
print(f"Connected components: {len(comp_sizes):,}")
print(f"Giant component: {comp_sizes[0]:,} nodes ({100*comp_sizes[0]/n_nodes:.1f}%)")
print(f"Component size > 1: {sum(1 for s in comp_sizes if s > 1):,}")
print(f"Isolated nodes: {sum(1 for s in comp_sizes if s == 1):,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGSIZE_WIDE, dpi=DPI)

ax1.loglog(range(1, len(comp_sizes) + 1), comp_sizes, marker=".", markersize=2,
           linewidth=0.8, color=COLOR_LEGIT)
ax1.set_xlabel("Component Rank")
ax1.set_ylabel("Component Size")
ax1.set_title("Component Size Distribution")

cumsum = np.cumsum(comp_sizes) / n_nodes * 100
ax2.plot(range(1, len(cumsum) + 1), cumsum, color=COLOR_LEGIT, linewidth=1.2)
ax2.axhline(y=90, color="gray", linestyle=":", linewidth=0.5, label="90% coverage")
ax2.axhline(y=100, color="black", linestyle="--", linewidth=0.5)
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Coverage (%)")
ax2.set_title("Cumulative Node Coverage")
ax2.legend(fontsize=7)

fig.suptitle("Figure 6: Graph connectivity structure", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "06_graph_connectivity.png", bbox_inches="tight")
plt.show()

The transaction graph has one giant component containing 72.2% of all active accounts. There are also 114K+ smaller components, with many being isolated nodes or tiny clusters. The giant component is where the interesting laundering patterns live: it contains the vast majority of accounts and transactions, so graph-based methods have a large connected structure to learn from. The presence of a dominant giant component is typical of financial transaction networks and validates the use of full-graph GNNs rather than subgraph-sampling approaches.

## 7. Laundering Pattern Structure: Fan-out vs Fan-in

In [ ]:
laundering_txn = txn[txn["is_laundering"] == 1]
src_laundering = laundering_txn.groupby("from_account").size()
dst_laundering = laundering_txn.groupby("to_account").size()

print(f"Max laundering fan-out: {src_laundering.max()} (one account sends to {src_laundering.max()} others)")
print(f"Max laundering fan-in:  {dst_laundering.max()} (one account receives from {dst_laundering.max()} others)")
print(f"Mean fan-out: {src_laundering.mean():.1f}")
print(f"Mean fan-in:  {dst_laundering.mean():.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGSIZE_WIDE, dpi=DPI)

ax1.hist(src_laundering.values, bins=50, color=COLOR_LAUNDER, alpha=0.7, density=True)
ax1.set_xlabel("Laundering Transactions Sent")
ax1.set_ylabel("Density")
ax1.set_title("Laundering Fan-out per Account")

ax2.hist(dst_laundering.values, bins=50, color=COLOR_LAUNDER, alpha=0.7, density=True)
ax2.set_xlabel("Laundering Transactions Received")
ax2.set_ylabel("Density")
ax2.set_title("Laundering Fan-in per Account")

fig.suptitle("Figure 7: Laundering transaction patterns per account", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "07_laundering_patterns.png", bbox_inches="tight")
plt.show()

The fan-out distribution is much more extreme than fan-in: one account sends 243 laundering transactions (smurfing, where a single source disperses many small transfers). Fan-in is capped at 31, suggesting receiving accounts are more diversified. This asymmetry (high fan-out, moderate fan-in) is consistent with the structuring/smurfing typology described in FATF documentation: a single account splits funds across many destination accounts to stay below reporting thresholds. This structural pattern is detectable via neighborhood aggregation in GNNs but invisible to flat ML models.

## 8. Entity Type Analysis

In [ ]:
accounts["entity_type"] = accounts["entity_name"].apply(
    lambda x: str(x).split("#")[0].strip() if isinstance(x, str) and "#" in x else (
        str(x)[:30] if isinstance(x, str) else "Unknown"
    )
)

entity_counts = accounts["entity_type"].value_counts()
print("Entity types found:")
print(entity_counts.to_string())

# Compute laundering rate per entity type
acct_to_type = accounts.set_index("account_id")["entity_type"].to_dict()
txn["src_type"] = txn["from_account"].map(acct_to_type).fillna("Orphan")
txn["dst_type"] = txn["to_account"].map(acct_to_type).fillna("Orphan")

top_entities = entity_counts.head(8).index
entity_launder_rates = {}
for etype in top_entities:
    etype_accts = set(accounts[accounts["entity_type"] == etype]["account_id"])
    etype_txn = txn[(txn["from_account"].isin(etype_accts)) |
                    (txn["to_account"].isin(etype_accts))]
    if len(etype_txn) > 0:
        entity_launder_rates[etype] = etype_txn["is_laundering"].mean() * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGSIZE_WIDE, dpi=DPI)

et_rates = pd.Series(entity_launder_rates).sort_values(ascending=False)
ax1.barh(et_rates.index, et_rates.values, color=COLOR_LAUNDER)
ax1.axvline(x=prevalence * 100, color="black", linestyle="--", linewidth=0.8)
ax1.set_xlabel("Laundering Rate (%)")
ax1.set_title("Laundering Rate by Entity Type (top 8)")

top_counts = entity_counts.head(8)
ax2.barh(top_counts.index, top_counts.values, color=COLOR_LEGIT)
ax2.set_xlabel("Number of Accounts")
ax2.set_title("Account Count by Entity Type (top 8)")

print("\nLaundering rate by entity type (top 8):")
for etype, rate in et_rates.items():
    print(f"  {etype}: {rate:.3f}%")

fig.suptitle("Figure 8: Entity type analysis", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "08_entity_types.png", bbox_inches="tight")
plt.show()

The dataset contains six entity types. Corporation, Partnership, and Sole Proprietorship dominate (98.4% of accounts). Country, Individual, and Direct entities are rare. Laundering rates vary across entity types. Corporation accounts have the highest laundering rate, which aligns with the FATF observation that corporate structures are commonly used for layering due to their ability to open multiple sub-accounts and process high transaction volumes.

## Summary of EDA Findings and Modeling Implications

| Finding | Implication for Modeling |
|---------|-------------------------|
| 0.102% laundering prevalence (1:981 ratio) | Use `pos_weight` in BCE loss; report AUC-PR and F1, not accuracy |
| Laundering rate varies across days (0.06% to 0.16%) | Chronological split is essential; random split leaks temporal information |
| Laundering accounts have 3.7x higher median degree (22 vs 6) | Graph structure contains discriminative signal beyond flat features |
| ACH payment format has 7.3x elevated laundering rate | Payment format is a strong univariate feature; one-hot encode it |
| Fan-out is highly asymmetric (max 243 out vs 31 in) | Directional neighborhood aggregation (GNNs) can capture smurfing patterns |
| Giant component covers 72.2% of nodes | Full-graph GNNs are feasible; most laundering happens in the connected core |
| Amount distributions are heavy-tailed and overlapping | Log1p transform amount; amount alone is insufficient for classification |
| Entity types differ in laundering rates | Entity type is a useful categorical feature for account-level node features |